# `ResultBase`

`nematics3d.classes.result_base.ResultBase` is the internal common base class for named algorithm results in Nematics3D. Users interact with the concrete result classes returned by Nematics3D functions rather than constructing `ResultBase` directly. The shared base lets those results expose related values under stable field names instead of requiring users to remember the order of a tuple, while preserving direct attribute access, a readable representation, and lightweight dictionary-style inspection.

## What `ResultBase` is for

Scientific calculations often produce several related outputs that must be interpreted together. For example, `q_diagonalize()` returns the scalar order parameter $S$, the director $\mathbf{n}$, diagnostic indices, and—when requested—the complete eigenvalues and eigenvectors. Returning these arrays as a positional tuple would make every caller remember which value occupies each position, and adding another output later could make that interface even harder to use safely.

`q_diagonalize()` therefore returns a `QDiagonalizationResult`, which is a concrete subclass of `ResultBase`. Its values are accessed by meaning, such as `result.S`, `result.n`, and `result.isotropic_indices`, rather than by numeric position. `ResultBase` supplies the behavior shared by this and future result types: discoverable field names, readable summaries, field descriptions, and a small dictionary-like interface. Users normally work with the concrete result returned by a function; they do not instantiate `ResultBase` directly.

## Setup

**For readers who are only interested in the tutorial, this section can be safely skipped.** The code cell below imports NumPy for constructing a small $Q$-tensor example and imports `nematics3d` through its public package interface.

In [15]:
import numpy as np

import nematics3d as n3d

## Minimal example

`ResultBase` objects are normally created and returned by Nematics3D functions rather than constructed directly by users. In this example, a uniaxial $Q$ tensor is constructed directly from a known scalar order parameter $S$ and director $\mathbf{n}$, then passed to `q_diagonalize()`. The returned object is a `QDiagonalizationResult`: a function-specific result class that inherits the common behavior defined by `ResultBase`.

In [16]:
input_director = np.array([1.0, 1.0, 1.0])
input_director /= np.linalg.norm(input_director)
input_scalar_order = 0.75

q_tensor = input_scalar_order * (
    np.outer(input_director, input_director) - np.eye(3) / 3
)
result = n3d.q_diagonalize(q_tensor)
result

QDiagonalizationResult: Q-tensor diagonalization
  S                 = 0.75,
  n                 = [-0.5774, -0.5774, -0.5774],
  isotropic_indices = [],
  eigenvalues       = None,
  eigenvectors      = None,

## Understanding the available fields

When a function returns an unfamiliar result object, the first useful step is to ask what fields it contains and what each field means. A concrete result class can provide a short description for every field. The subclass owns this result-specific documentation, while `ResultBase` supplies a consistent interface for displaying it. This keeps the explanations attached to the result type and makes them available during interactive work without requiring the user to locate the class definition.

`show_readable_attrs()` displays all declared field names and, by default, their descriptions. Pass `is_desc=False` when only the names are needed. After identifying a field of interest, `show_attr_doc("field_name")` displays its individual description and raises `KeyError` when that field does not exist. In the example below, `S` is merely one field documented by the concrete `QDiagonalizationResult` subclass; the same methods apply to every `ResultBase` result.

In [17]:
result.show_readable_attrs()
result.show_attr_doc("S")

[INFO]
    <show_readable_attrs> 
    - S
        Scalar order: 3/2 times the largest eigenvalue.
    - n
        Unit eigenvector for the largest eigenvalue.
    - isotropic_indices
        Coordinate indices of points handled as numerically isotropic.
    - eigenvalues
        Descending eigenspectrum when biaxial output is requested.
    - eigenvectors
        Eigenvector columns matching the descending eigenvalues.
[INFO]
    <show_attr_doc> 
    Scalar order: 3/2 times the largest eigenvalue.


## Accessing fields by name

Every concrete `ResultBase` subclass declares its own set of dataclass fields. `ResultBase` does not prescribe those field names or the kinds of values they contain; it makes each declared field available through two familiar interfaces.

A result can be used like an ordinary Python object, with a field read as `result.field_name`. The same value can also be read with dictionary-style subscription as `result["field_name"]`. A dictionary-style key is always exactly the same string as the corresponding attribute name; there is no separate key naming system. In the running example, `result.S` and `result["S"]` therefore access the same field declared by `QDiagonalizationResult`.

In [18]:
scalar_order_by_attribute = result.S
scalar_order_by_key = result["S"]

scalar_order_by_attribute, scalar_order_by_key

(np.float64(0.75), np.float64(0.75))

## Dictionary-like inspection

`ResultBase` supports many of the read-oriented operations commonly used with dictionaries. `keys()`, `values()`, and `items()` return tuples in dataclass declaration order; `get()` reads a field with an optional fallback; and `asdict()` creates a shallow dictionary containing all fields. The `in` operator tests whether a field name exists, iteration yields field names, and `len()` returns the number of fields.

These methods make interactive inspection and generic result-processing code convenient, but a `ResultBase` object is not a mutable dictionary: it does not support assigning or deleting fields by key. In particular, `asdict()` is shallow, so it creates a new dictionary without copying arrays or other objects stored in the result.

In [19]:
dictionary_features = {
    "keys": result.keys(),
    "values": result.values(),
    "items": result.items(),
    "get": result.get("S"),
    "get with fallback": result.get("missing_field", "not available"),
    "contains": "S" in result,
    "iterated keys": tuple(result),
    "length": len(result),
    "shallow dictionary": result.asdict(),
}

dictionary_features

{'keys': ('S',
  'n',
  'isotropic_indices',
  'eigenvalues',
  'eigenvectors'),
 'values': (np.float64(0.75),
  array([-0.57735027, -0.57735027, -0.57735027]),
  array([], shape=(0, 0), dtype=int64),
  None,
  None),
 'items': (('S', np.float64(0.75)),
  ('n', array([-0.57735027, -0.57735027, -0.57735027])),
  ('isotropic_indices', []),
  ('eigenvalues', None),
  ('eigenvectors', None)),
 'get': np.float64(0.75),
 'get with fallback': 'not available',
 'contains': True,
 'iterated keys': ('S',
  'n',
  'isotropic_indices',
  'eigenvalues',
  'eigenvectors'),
 'length': 5,
 'shallow dictionary': {'S': np.float64(0.75),
  'n': array([-0.57735027, -0.57735027, -0.57735027]),
  'isotropic_indices': [],
  'eigenvalues': None,
  'eigenvectors': None}}

## Readable representation

Every `ResultBase` object has a structured text representation intended for interactive work, logs, and quick inspection. The first line identifies the concrete result class and, when the subclass provides one, a short human-readable result name. The remaining lines show all declared fields in declaration order, with their names aligned so the structure can be scanned quickly.

Displaying a result only formats values that have already been stored; it does not recompute the underlying calculation. Large or structured values are represented compactly according to the repository's common formatting rules, so this summary is useful for orientation rather than a replacement for accessing the original fields.

In [21]:
result

QDiagonalizationResult: Q-tensor diagonalization
  S                 = 0.75,
  n                 = [-0.5774, -0.5774, -0.5774],
  isotropic_indices = [],
  eigenvalues       = None,
  eigenvectors      = None,

# For developers

The remaining sections explain how Nematics3D repository developers define and document new internal result types. Users who only need to inspect results returned by Nematics3D functions can stop here.

## Call relationships inside Nematics3D

**This section is intended for developers. Regular users can safely skip it.**

`ResultBase` itself is not directly instantiated inside Nematics3D. The following concrete subclasses and their direct construction sites were found in `src/nematics3d`:

- `QDiagonalizationResult`
  - `q_diagonalize()`
- `OBBFit`
  - `_obb_fit_in_axes()`
- `TriangulationQuality`
  - `check_triangulation_quality()`
- `NMLIterationResult`
  - `nml_principal_plane_analysis()`
- `NMLPrincipalPlaneResult`
  - `nml_principal_plane_analysis()`
- `FourierResult`
  - `act_fourier()`
- `RadialSpectrumResult`
  - `_radial_spectrum_result_1d()`
  - `_radial_spectrum_result_binned()`
- `CorrelationResult`
  - `_correlation_result_from_fourier()`
- `DistanceCorrelationResult`
  - `_distance_result_1d()`
  - `_distance_result_binned()`
- `ThresholdRelaxationResult`
  - `_threshold_result()`
- `FitRelaxationResult`
  - `_empty_fit_result()`
  - `_fit_decay_result()`
- `RelaxationLengthResult`
  - `act_relaxation_length()`
- `OmegaResult`
  - `QPlanePolar.act_calc_omega()`
- `SpatialDerivativeInfo`
  - `_helper_spatial_derivative_info()`
- `GaussianSmoothInfo`
  - `_helper_gaussian_smooth_info()`

## Defining a `ResultBase` subclass

**This section is intended for developers. Regular users can safely skip it.**

Developers introduce a new result type by combining `ResultBase` with a dataclass. The annotated dataclass fields define the stored values, their access names, and their order throughout the inherited interface. `repr=False` is required when the subclass should use the structured representation supplied by `ResultBase`; `slots=True` and `frozen=True` are recommended for lightweight result containers whose fields should not change after creation.

Two optional class variables add human-readable metadata without becoming result fields. `__result_name__` labels the result as a whole, while `__field_docs__` maps field names to the descriptions displayed by the inspection methods. Because they are annotated with `ClassVar`, neither appears in `keys()`, `items()`, or the result representation. Field names must not collide with inherited interface names such as `keys`, `values`, `items`, `get`, or `asdict`, because a dataclass field with the same name would hide that method on the instance. The minimal example below uses $S$ and $\mathbf{n}$ only to remain consistent with the running diagonalization example.

In [ ]:
from dataclasses import dataclass
from typing import ClassVar


@dataclass(slots=True, frozen=True, repr=False)
class MinimalDiagonalizationResult(n3d.ResultBase):
    __result_name__: ClassVar[str] = "minimal diagonalization"
    __field_docs__: ClassVar[dict[str, str]] = {
        "S": "Scalar order parameter.",
        "n": "Dominant nematic director.",
    }

    S: np.ndarray
    n: np.ndarray


minimal_result = MinimalDiagonalizationResult(S=result.S, n=result.n)
minimal_result